# 06 · Topic discovery: embeddings → clusters → LLM-named topics

What do people talk about, with no labels? Three steps:

1. **Embed** each conversation (`topics.embed`, any pydantic-ai embedding model string; cached).
2. **Cluster** the normalized vectors with KMeans, choosing *k* by silhouette (`topics.cluster`).
3. **Name** each cluster with an LLM from its most central examples (`topics.label_clusters`, a normal typed `Task`).

Evaluation: the synthetic chats from notebook 04 carry their true **domain** in `meta`, so we measure how well clusters recover it (NMI / ARI, 1 = perfect) and compare embedding models. The tweets section is exploratory.

In [ ]:
%load_ext autoreload
%autoreload 2

import logging
import os

import matplotlib.pyplot as plt
import pandas as pd
from dotenv import load_dotenv
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

from genaianalysis.schema import load_jsonl
from genaianalysis.topics import cluster, embed, label_clusters
from genaianalysis.utils.paths import data_processed_dir, project_dir

load_dotenv(project_dir(".env"))
os.environ.setdefault("PYDANTIC_AI_NO_BANNER", "1")
logging.basicConfig(level=logging.ERROR)
pd.set_option("display.max_colwidth", 160)

EMBEDDERS = ["google:gemini-embedding-001", "google:gemini-embedding-2"]
NAMER = "google:gemini-flash-lite-latest"
RPM = 12

## 1 · Synthetic chats: can clusters recover the business domain?
Embedding the customer's messages only keeps the agent's boilerplate ("¡Con gusto te ayudo!") from dominating the vectors.

In [ ]:
chats = load_jsonl(data_processed_dir("synthetic_es_45.jsonl"))
domain = [c.meta["domain"] for c in chats]
customer_text = ["\n".join(m.text for m in c.messages if m.sender == "customer") for c in chats]
print(len(chats), "chats,", len(set(domain)), "domains")

rows, fits = [], {}
for model in EMBEDDERS:
    X = await embed(customer_text, model)
    labels, sil = cluster(X, k_range=range(2, 13))
    fixed, _ = cluster(X, k=len(set(domain)))  # oracle k, isolates embedding quality from k choice
    fits[model] = (X, labels)
    rows.append({
        "embedder": model, "chosen_k": len(set(labels)),
        "nmi": normalized_mutual_info_score(domain, labels), "ari": adjusted_rand_score(domain, labels),
        "nmi_oracle_k": normalized_mutual_info_score(domain, fixed),
    })
pd.DataFrame(rows)

In [ ]:
best = max(rows, key=lambda r: r["nmi"])["embedder"]
X, labels = fits[best]
pd.crosstab(pd.Series(domain, name="domain"), pd.Series(labels, name="cluster"))

In [ ]:
topics = await label_clusters(chats, labels, NAMER, X=X, language="Spanish", rpm=RPM)
topics

## 2 · Tweets (exploratory)
No topic labels exist; read the names and the central examples to judge.

In [ ]:
tweets = load_jsonl(data_processed_dir("tweets_spanish_test_100.jsonl"))
Xt = await embed([c.messages[0].text for c in tweets], best)
tlabels, tsil = cluster(Xt, k_range=range(3, 13))
ax = tsil.plot(x="k", y="silhouette", marker="o", figsize=(5, 3), title="silhouette by k")

In [ ]:
tweet_topics = await label_clusters(tweets, tlabels, NAMER, X=Xt, language="Spanish", rpm=RPM)
tweet_topics

In [ ]:
xy = PCA(n_components=2, random_state=0).fit_transform(Xt)
names = dict(zip(tweet_topics["cluster"], tweet_topics["name"], strict=True))
fig, ax = plt.subplots(figsize=(7, 5))
for c in sorted(set(tlabels)):
    m = tlabels == c
    ax.scatter(xy[m, 0], xy[m, 1], s=18, label=f"{c}: {names.get(c, '?')}")
ax.set(title="Tweets, PCA of embeddings", xticks=[], yticks=[])
ax.legend(fontsize=7, loc="best");

## Findings
Record them in `docs/research.md`: NMI/ARI per embedder, chosen k vs true k, and whether clusters follow topic or tone.